# Step 2b — TOD Review Reconciliation

Reconciles the fresh `SB79_tod_review.xlsx` produced by Step 2 with the
previously-reviewed workbook (`REVIEW_XLSX` in `config.py`) to carry forward
any `manual_station_assignment` decisions that remain valid.

Run this notebook **only** when Step 2 has been re-run after new stops or
access points were added, making the existing reviewed workbook stale.

Output is a new dated workbook (`SB79_tod_review_YYYY_MM_DD.xlsx`) written to
`DATA_DIR` that becomes the starting point for the next manual review cycle.

> **Pipeline order:** Run after `2_tod_stop_and_access_assignment.ipynb`
> and before manual review → `3_tod_station_assignment_reintegration.ipynb`.
>
> **Inputs (both set in `config.py`):**
> - `REVIEW_XLSX_OUTPUT` — fresh Step 2 output; authoritative ID universe.
> - `REVIEW_XLSX` — stale reviewed workbook containing prior manual overrides.
>
> **Carry-over rules:**
> - Only rows with `assignment_status == 'manual_station_assignment'` are
>   carried from the stale workbook.
> - `conflict` and `no_match` rows in the stale workbook are **discarded** —
>   the fresh spatial assignment from Step 2 is used instead.
> - Carried-over `station_id` values that no longer exist in the TOD station
>   universe are downgraded to `conflict` and flagged for re-review.
>
> **After running this notebook:**
> 1. Update `REVIEW_XLSX` in `config.py` to point to the new dated output file.
> 2. Open the file and resolve any remaining `conflict` or `no_match` rows.
> 3. Run `3_tod_station_assignment_reintegration.ipynb`.
>
> All data source paths are defined in `config.py`.

In [1]:
import datetime
import geopandas as gpd
import openpyxl
import pandas as pd
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from openpyxl.worksheet.datavalidation import DataValidation
from config import *

In [2]:
# All paths and layer names are imported from config.py.
# REVIEW_XLSX_OUTPUT — fresh Step 2 output (authoritative ID universe)
# REVIEW_XLSX        — stale reviewed workbook (prior manual overrides)
today_str = datetime.date.today().strftime("%Y_%m_%d")
reconciled_output = DATA_DIR / f"SB79_tod_review_{today_str}.xlsx"

print(f"Fresh Step 2 output (new):   {REVIEW_XLSX_OUTPUT}")
print(f"Stale reviewed workbook:     {REVIEW_XLSX}")
print(f"Reconciled output (→ today): {reconciled_output}")

if not REVIEW_XLSX_OUTPUT.exists():
    raise FileNotFoundError(
        f"Fresh Step 2 output not found: {REVIEW_XLSX_OUTPUT}\n"
        "Run 2_tod_stop_and_access_assignment.ipynb first."
    )
if not REVIEW_XLSX.exists():
    raise FileNotFoundError(
        f"Stale reviewed workbook not found: {REVIEW_XLSX}\n"
        "Update REVIEW_XLSX in config.py to point to the previously-reviewed file."
    )

Fresh Step 2 output (new):   /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review.xlsx
Stale reviewed workbook:     /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review_2026_03_06.xlsx
Reconciled output (→ today): /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review_2026_03_20.xlsx


In [3]:
# Load both workbooks — all sheets, all rows, all columns as strings
new_stops_df = pd.read_excel(REVIEW_XLSX_OUTPUT, sheet_name=REVIEW_STOPS_SHEET, dtype=str)
new_access_df = pd.read_excel(REVIEW_XLSX_OUTPUT, sheet_name=REVIEW_ACCESS_PTS_SHEET, dtype=str)

stale_stops_df = pd.read_excel(REVIEW_XLSX, sheet_name=REVIEW_STOPS_SHEET, dtype=str)
stale_access_df = pd.read_excel(REVIEW_XLSX, sheet_name=REVIEW_ACCESS_PTS_SHEET, dtype=str)

print(f"Fresh workbook:  {REVIEW_XLSX_OUTPUT.name}")
print(f"  stops:         {len(new_stops_df)} rows")
print(f"  access points: {len(new_access_df)} rows")
print(f"\nStale workbook:  {REVIEW_XLSX.name}")
print(f"  stops:         {len(stale_stops_df)} rows")
print(f"  access points: {len(stale_access_df)} rows")

# Filter stale sheets to manual assignments only — these are the only rows to carry forward
MANUAL_STATUS = "manual_station_assignment"
stale_stops_manual = stale_stops_df[
    stale_stops_df["assignment_status"].str.strip() == MANUAL_STATUS
].copy()
stale_access_manual = stale_access_df[
    stale_access_df["assignment_status"].str.strip() == MANUAL_STATUS
].copy()

print(f"\nManual assignments in stale workbook:")
print(f"  stops:         {len(stale_stops_manual)}")
print(f"  access points: {len(stale_access_manual)}")

Fresh workbook:  SB79_tod_review.xlsx
  stops:         769 rows
  access points: 1160 rows

Stale workbook:  SB79_tod_review_2026_03_06.xlsx
  stops:         757 rows
  access points: 1190 rows

Manual assignments in stale workbook:
  stops:         24
  access points: 94


## Validate carried-over station IDs

Load the current TOD station universe and check whether each
`manual_station_assignment` row from the stale workbook still references a
valid station.  Rows whose `station_id` no longer exists are downgraded to
`conflict` so they appear in the review queue rather than being silently
carried forward with a stale ID.

In [4]:
tod_stations_dev = gpd.read_file(TOD_DATABASE_GPKG, layer=GPKG_TOD_STATIONS_DEV_LAYER)
valid_station_ids = set(tod_stations_dev["station_id"].dropna().astype(str))
print(f"Loaded {len(tod_stations_dev)} TOD stations from '{GPKG_TOD_STATIONS_DEV_LAYER}'")
print(f"{len(valid_station_ids)} unique valid station_ids")


def _split_on_validity(manual_df, id_col):
    """Split a manual-assignment DataFrame into (valid, invalid) based on
    whether station_id exists in the current TOD station universe.

    Returns two DataFrames.  `invalid` rows have their assignment_status
    set to 'conflict' so the reviewer sees them in the output workbook.
    """
    if manual_df.empty:
        return manual_df.copy(), manual_df.copy()
    df = manual_df.copy()
    df["station_id"] = df["station_id"].astype(str).str.strip()
    is_valid = df["station_id"].isin(valid_station_ids)
    invalid = df[~is_valid].copy()
    invalid["assignment_status"] = "conflict"
    return df[is_valid], invalid


stops_carryover_valid, stops_carryover_invalid = _split_on_validity(
    stale_stops_manual, "stop_id"
)
access_carryover_valid, access_carryover_invalid = _split_on_validity(
    stale_access_manual, "access_id"
)

print(f"\nStop carry-over validation:")
print(f"  Valid station_id:   {len(stops_carryover_valid)}")
print(f"  Invalid station_id: {len(stops_carryover_invalid)} (will be downgraded to 'conflict')")
if not stops_carryover_invalid.empty:
    print(stops_carryover_invalid[["stop_id", "station_id"]].to_string(index=False))

print(f"\nAccess point carry-over validation:")
print(f"  Valid station_id:   {len(access_carryover_valid)}")
print(f"  Invalid station_id: {len(access_carryover_invalid)} (will be downgraded to 'conflict')")
if not access_carryover_invalid.empty:
    print(access_carryover_invalid[["access_id", "station_id"]].to_string(index=False))

Loaded 432 TOD stations from 'tod_stations_dev'
432 unique valid station_ids

Stop carry-over validation:
  Valid station_id:   24
  Invalid station_id: 0 (will be downgraded to 'conflict')

Access point carry-over validation:
  Valid station_id:   94
  Invalid station_id: 0 (will be downgraded to 'conflict')


## Reconcile stops and access points

For each sheet, the reconciliation applies the following rules in order:

| Record state | Action |
|---|---|
| In new sheet; matched a valid carry-over row | Overwrite `station_id` + `assignment_status` from stale; all other columns from fresh Step 2 output |
| In new sheet; matched an invalid carry-over row | Overwrite `assignment_status` = `conflict`; `station_id` kept from stale for reference |
| In new sheet; not in stale sheet at all | Keep fresh Step 2 spatial assignment unchanged |
| In stale sheet only (row dropped from new universe) | Reported below; not written to output |

In [5]:
def reconcile(
    new_df: pd.DataFrame,
    carryover_valid: pd.DataFrame,
    carryover_invalid: pd.DataFrame,
    id_col: str,
) -> tuple[pd.DataFrame, dict]:
    """Apply carry-over rules and return the reconciled DataFrame plus a
    stats dict with keys: carried, carried_as_conflict, new_records, dropped.
    """
    result = new_df.copy()

    # Build lookup maps keyed on id_col → (station_id, assignment_status)
    valid_map = (
        carryover_valid.set_index(id_col)[["station_id", "assignment_status"]]
        if not carryover_valid.empty
        else pd.DataFrame(columns=[id_col, "station_id", "assignment_status"]).set_index(id_col)
    )
    invalid_map = (
        carryover_invalid.set_index(id_col)[["station_id", "assignment_status"]]
        if not carryover_invalid.empty
        else pd.DataFrame(columns=[id_col, "station_id", "assignment_status"]).set_index(id_col)
    )

    result = result.set_index(id_col)

    # Apply valid carry-overs
    valid_matches = result.index.isin(valid_map.index)
    result.loc[valid_matches, "station_id"] = valid_map.loc[
        valid_map.index.isin(result.index), "station_id"
    ]
    result.loc[valid_matches, "assignment_status"] = MANUAL_STATUS

    # Apply invalid carry-overs (overwrite status only; keep stale station_id for reference)
    invalid_matches = result.index.isin(invalid_map.index)
    result.loc[invalid_matches, "station_id"] = invalid_map.loc[
        invalid_map.index.isin(result.index), "station_id"
    ]
    result.loc[invalid_matches, "assignment_status"] = "conflict"

    result = result.reset_index()

    all_stale_manual_ids = set(valid_map.index) | set(invalid_map.index)
    new_ids = set(new_df[id_col])

    stats = {
        "carried": int(valid_matches.sum()),
        "carried_as_conflict": int(invalid_matches.sum()),
        "new_records": int(len(new_ids - all_stale_manual_ids - set(new_df.loc[
            new_df[id_col].isin(all_stale_manual_ids), id_col
        ]))),
        "dropped": sorted(all_stale_manual_ids - new_ids),
    }
    # Recalculate new_records cleanly
    stale_all_ids = set(pd.concat([
        carryover_valid[[id_col]], carryover_invalid[[id_col]]
    ], ignore_index=True)[id_col]) if (not carryover_valid.empty or not carryover_invalid.empty) else set()
    stats["new_records"] = int(len(new_ids - stale_all_ids))

    return result, stats


reconciled_stops_df, stops_stats = reconcile(
    new_stops_df, stops_carryover_valid, stops_carryover_invalid, "stop_id"
)
reconciled_access_df, access_stats = reconcile(
    new_access_df, access_carryover_valid, access_carryover_invalid, "access_id"
)

In [6]:
# ── Reconciliation summary ────────────────────────────────────────────────────
print("=== RECONCILIATION SUMMARY ===")

for label, stats, id_col in [
    ("Stops", stops_stats, "stop_id"),
    ("Access points", access_stats, "access_id"),
]:
    print(f"\n{label}:")
    print(f"  Carried forward (valid):           {stats['carried']}")
    print(f"  Carried as conflict (invalid ID):  {stats['carried_as_conflict']}")
    print(f"  New records (not in stale sheet):  {stats['new_records']}")
    if stats["dropped"]:
        print(f"  Dropped (in stale, not in new):    {len(stats['dropped'])}")
        print(f"    {id_col} values dropped:")
        for v in stats["dropped"]:
            print(f"      {v}")
    else:
        print(f"  Dropped (in stale, not in new):    0")

=== RECONCILIATION SUMMARY ===

Stops:
  Carried forward (valid):           24
  Carried as conflict (invalid ID):  0
  New records (not in stale sheet):  745
  Dropped (in stale, not in new):    0

Access points:
  Carried forward (valid):           88
  Carried as conflict (invalid ID):  0
  New records (not in stale sheet):  1072
  Dropped (in stale, not in new):    6
    access_id values dropped:
      70242
      geom:37.32833,-121.81117
      geom:37.33635,-121.89164
      geom:37.33650,-121.89018
      geom:37.35484,-121.93620
      geom:37.78810,-122.39856


## Write reconciled review workbook

Writes the reconciled sheets to a new dated workbook using the same formatting
and data-validation conventions as Step 2.

In [7]:
# Review column sets — must match Step 2's STOPS_REVIEW_COLS / ACCESS_REVIEW_COLS
STOPS_REVIEW_COLS = [
    "stop_id", "stop_name", "tod_tier", "station_id", "assignment_status", "buffer_distance_m"
]
ACCESS_REVIEW_COLS = [
    "access_id", "access_point_name", "agency", "station_id", "assignment_status", "buffer_distance_m"
]
STATUS_VALUES = '"conflict,no_match,manual_station_assignment"'

HEADER_FILL = PatternFill(start_color="4472C4", end_color="4472C4", fill_type="solid")
HEADER_FONT = Font(bold=True, color="FFFFFF")


def _write_review_sheet(ws, df, cols):
    """Write df columns to ws with a styled header, data validation on
    assignment_status, frozen header row, and auto-fitted column widths."""
    for col_idx, col_name in enumerate(cols, start=1):
        cell = ws.cell(row=1, column=col_idx, value=col_name)
        cell.font = HEADER_FONT
        cell.fill = HEADER_FILL
        cell.alignment = Alignment(horizontal="center")

    df_out = df[[c for c in cols if c in df.columns]].copy()
    df_out = df_out.astype(object).where(df_out.notna(), other=None)
    for row_idx, row in enumerate(df_out.itertuples(index=False), start=2):
        for col_idx, value in enumerate(row, start=1):
            ws.cell(row=row_idx, column=col_idx, value=value)

    status_col_idx = cols.index("assignment_status") + 1
    status_col_letter = get_column_letter(status_col_idx)
    dv = DataValidation(
        type="list",
        formula1=STATUS_VALUES,
        showDropDown=False,
        showErrorMessage=True,
        errorTitle="Invalid value",
        error="Please select: conflict, no_match, or manual_station_assignment",
    )
    ws.add_data_validation(dv)
    dv.sqref = f"{status_col_letter}2:{status_col_letter}{len(df_out) + 1}"

    ws.freeze_panes = "A2"

    for col_idx, col_name in enumerate(cols, start=1):
        col_letter = get_column_letter(col_idx)
        max_len = len(col_name)
        for row_idx in range(2, len(df_out) + 2):
            val = ws.cell(row=row_idx, column=col_idx).value
            if val is not None:
                max_len = max(max_len, len(str(val)))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 60)


wb = openpyxl.Workbook()
ws_stops = wb.active
ws_stops.title = REVIEW_STOPS_SHEET
_write_review_sheet(ws_stops, reconciled_stops_df, STOPS_REVIEW_COLS)

ws_access = wb.create_sheet(title=REVIEW_ACCESS_PTS_SHEET)
_write_review_sheet(ws_access, reconciled_access_df, ACCESS_REVIEW_COLS)

wb.save(reconciled_output)

print(f"Reconciled review workbook written: {reconciled_output}")
print(f"  Sheet '{REVIEW_STOPS_SHEET}':       {len(reconciled_stops_df)} rows")
print(f"  Sheet '{REVIEW_ACCESS_PTS_SHEET}': {len(reconciled_access_df)} rows")
print()
print("Next steps:")
print(f"1. Update REVIEW_XLSX in config.py to point to:")
print(f"     DATA_DIR / \"{reconciled_output.name}\"")
print(f"2. Open the file and resolve any remaining 'conflict' or 'no_match' rows:")
print(f"   - Set assignment_status = 'manual_station_assignment'")
print(f"   - Enter the correct station_id")
print(f"3. Save the workbook, then run 3_tod_station_assignment_reintegration.ipynb")

Reconciled review workbook written: /Users/jcroff/Library/CloudStorage/Box-Box/DSA Projects/Spatial Analysis and Mapping/SB79 Transit Oriented Development/Data/SB79_tod_review_2026_03_20.xlsx
  Sheet 'stops':       769 rows
  Sheet 'access_points': 1160 rows

Next steps:
1. Update REVIEW_XLSX in config.py to point to:
     DATA_DIR / "SB79_tod_review_2026_03_20.xlsx"
2. Open the file and resolve any remaining 'conflict' or 'no_match' rows:
   - Set assignment_status = 'manual_station_assignment'
   - Enter the correct station_id
3. Save the workbook, then run 3_tod_station_assignment_reintegration.ipynb
